Activate the Environment containing specific package versions 

In [45]:
using Pkg
Pkg.activate(raw"C:\Users\SurfacePro8\Documents\Studium\Master Thesis\Theoretical Model")

  Activating project at `C:\Users\SurfacePro8\Documents\Studium\Master Thesis\Theoretical Model`


Include the main simulation loop and dependencies

In [46]:
include("Main Simulation.jl")

run_simulation (generic function with 1 method)

Define a parameter construct for the model configuration. \
Parameters that can be changed include:
- Number of species
- Number of wells 
- Carrying capacity 
- Duration of simulation 
- Mutation rate \
and more 

In [47]:
p = const_params(no_species = 2, K = 1e7)

Params(2, 1, 200, 1.0, 100.0, 1.0, 1.0, 5.0e-6, 5.0, [52.0, 40.0], 0.35, 5.0e-7, 0.002, 1.0e7, 0.0033333333333333335, 15, [1.0, 1.0234114021054532, 1.0473708979594496, 1.0718913192051278, 1.0969857978923836, 1.1226677735108137, 1.1489510001873091, 1.1758495540521567, 1.2033778407775895, 1.2315506032928256  …  81.19844993184013, 83.09941949353396, 85.04489341802677, 87.03591361485161, 89.0735463861044, 91.15888299750823, 93.29304026284686, 95.47716114208056, 97.71241535346496, 100.0], [0.01332975347230664, 0.01732867951399863], 21599, 1440, [0.05206783908494098 0.0519510980447802 … 3.1595135409373083e-40 1.2960316488116356e-40; 0.04938546871745239 0.04949644444662628 … 7.289196169001087e-40 3.003479483798981e-40; … ; 3.0034794837989814e-40 7.289196169001088e-40 … 0.049496444446626285 0.0493854687174524; 1.2960316488116356e-40 3.1595135409373083e-40 … 0.0519510980447802 0.05206783908494098])

Define a model configuration. \
The model configuration includes 
- the functions that should be used in the main simulation loop (how to calculate the growth, death, crowding etc.)
- if the strains shoudl interact or the wells should be pooled before dilutions
- the metrics that should be recorded during the simulation (survival time, entropy, etc.)
- the parameters that should be used (number of species, mutation rate, growth rates etc.)
\
\
Below we define two configurations: a monoculture configuration and a co-culture configuration. 

In [48]:
mono_config = ModelConfig(
    growth_fn = growth_rate,
    death_fn = death_rate,
    interaction = NoInteraction(),
    interaction_fn = growth_interaction,
    dilution = NoPooling(),
    mutation_fn = mutate_newborns!,
    crowding_fn = crowd_growth,
    drug_schedule = logistic_schedule,
    record_fn = record!,
    metrics = [Every(ResistantMetric(), 1.0), 
    Every(LineageSurvivalMetric(p), 1.0)], 
    params = const_params()
)

co_config = ModelConfig(
    growth_fn = growth_rate,
    death_fn = death_rate,
    interaction = Interaction(),
    interaction_fn = growth_interaction,
    dilution = NoPooling(),
    mutation_fn = mutate_newborns!,
    crowding_fn = crowd_growth,
    drug_schedule = logistic_schedule,
    record_fn = record!,
    metrics = [Every(ResistantMetric(), 1.0), 
    Every(LineageSurvivalMetric(p), 1.0)], 
    params = p
)

ModelConfig(Main.growth_rate, Main.death_rate, Interaction(), Main.growth_interaction, NoPooling(), Main.mutate_newborns!, Main.crowd_growth, Main.logistic_schedule, Main.record!, Every[Every{ResistantMetric}(ResistantMetric(Int64[], Int64[], Float64[], Float64[], Float64[], Float64[]), 1.0), Every{LineageSurvivalMetric}(LineageSurvivalMetric(Union{Nothing, Float64}[nothing; nothing; … ; nothing; nothing;;], Union{Nothing, Float64}[nothing; nothing; … ; nothing; nothing;;], Bool[0; 0; … ; 0; 0;;]), 1.0)], Params(2, 1, 200, 1.0, 100.0, 1.0, 1.0, 5.0e-6, 5.0, [52.0, 40.0], 0.35, 5.0e-7, 0.002, 1.0e7, 0.0033333333333333335, 15, [1.0, 1.0234114021054532, 1.0473708979594496, 1.0718913192051278, 1.0969857978923836, 1.1226677735108137, 1.1489510001873091, 1.1758495540521567, 1.2033778407775895, 1.2315506032928256  …  81.19844993184013, 83.09941949353396, 85.04489341802677, 87.03591361485161, 89.0735463861044, 91.15888299750823, 93.29304026284686, 95.47716114208056, 97.71241535346496, 100.0], 

Below is a typical simulation loop. \
The function run_simulation initializes the metrics, runs the main simulation and outputs the recorded quantities. \
These can afterwards be stored in a DataFrame containing the simulation index. 
\
\
Here we store the Extinction Time and Establishment Probability from the output named tuple _results_. 

In [ ]:
using DataFrames

rows = NamedTuple[]

nreps = 10
Threads.@threads for r in 1:nreps

    results = run_simulation(co_counts0, co_config)
    row = (Model = "co int", 
    Extinction_Time = results.extinction_time, 
    P_est = p_est(results.LineageSurvivalMetric))

    push!(rows, merge(row, (; Replicate = r, results.ResistantMetric.drug_conc)))
end

df = DataFrame(rows)

Progress: 100%|█████████████████████████████████████████| Time: 0:00:02
Progress: 100%|█████████████████████████████████████████| Time: 0:00:01
Progress: 100%|█████████████████████████████████████████| Time: 0:00:01
Progress: 100%|█████████████████████████████████████████| Time: 0:00:01
Progress: 100%|█████████████████████████████████████████| Time: 0:00:01
Progress: 100%|█████████████████████████████████████████| Time: 0:00:01
Progress: 100%|█████████████████████████████████████████| Time: 0:00:01
Progress: 100%|█████████████████████████████████████████| Time: 0:00:01
Progress: 100%|█████████████████████████████████████████| Time: 0:00:02
Progress: 100%|█████████████████████████████████████████| Time: 0:00:01

In [ ]:
using CairoMakie

df.Extinction_Time = df.Extinction_Time ./ 24.0

fig = Figure()

dens_ax = Axis(fig[1, 1])
box_ax = Axis(fig[2, 1])

density!(dens_ax, df.Extinction_Time)
boxplot!(box_ax, fill(1, length(df.P_est)), df.P_est)

fig